# E-Commerce Customer Churn & Retention Analytics
**Dataset:** E-Commerce Customer Data for Behavior Analysis (Kaggle)  
**Author:** Aqsa Fatima  
**Date:** 2024

---
## Project Flow
1. Setup & Imports
2. Data Loading & Initial Inspection
3. Data Cleaning
4. Exploratory Data Analysis (EDA)
5. Key Performance Indicators (KPIs)
6. Customer Segmentation & Behaviour Analysis
7. Churn Analysis
8. Machine Learning — Churn Prediction
9. Business Insights & Recommendations

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print('All libraries loaded successfully.')

## 2. Data Loading & Initial Inspection

In [ ]:
df = pd.read_csv('ecommerce_customer_data_large.csv')
print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(5)

In [ ]:
print('--- Column Data Types ---')
print(df.dtypes)
print()
print('--- Missing Values ---')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})[missing > 0])
print()
print(f'Duplicate rows: {df.duplicated().sum()}')

In [ ]:
print('--- Churn Distribution (Transaction Level) ---')
print(df['Churn'].value_counts())
print()
print('--- Unique Customers ---')
print(f'Total unique customers: {df["Customer ID"].nunique():,}')
print()
print('--- Product Categories ---')
print(df['Product Category'].value_counts())
print()
print('--- Payment Methods ---')
print(df['Payment Method'].value_counts())
print()
print('--- Gender ---')
print(df['Gender'].value_counts())

In [ ]:
print('--- Numerical Summary ---')
df[['Product Price', 'Quantity', 'Total Purchase Amount', 'Age', 'Returns']].describe().round(2)

**Observations:**
- 250,000 transactions for 49,661 unique customers.
- `Customer Age` and `Age` are identical columns — one will be dropped.
- `Returns` has ~18.95% missing values; these will be imputed with the median.
- No fully duplicate rows.
- `Churn` is a binary flag at the customer level (same value for all transactions of a customer).
- Purchase dates span January 2020 – September 2023.

## 3. Data Cleaning

In [ ]:
# 3.1 Parse datetime
df['Purchase Date'] = pd.to_datetime(df['Purchase Date'])

# 3.2 Drop the redundant 'Customer Age' column (identical to 'Age')
df.drop(columns=['Customer Age'], inplace=True)

# 3.3 Impute missing 'Returns' with median (0 or 1)
returns_median = int(df['Returns'].median())
df['Returns'] = df['Returns'].fillna(returns_median).astype(int)

# 3.4 Strip whitespace from string columns
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

# 3.5 Add useful derived columns
df['Year']       = df['Purchase Date'].dt.year
df['Month']      = df['Purchase Date'].dt.month
df['YearMonth']  = df['Purchase Date'].dt.to_period('M')
df['DayOfWeek']  = df['Purchase Date'].dt.dayofweek   # 0=Monday
df['Age Group']  = pd.cut(
    df['Age'],
    bins=[17, 25, 35, 45, 55, 70],
    labels=['18-25', '26-35', '36-45', '46-55', '56-70']
)

print('Cleaning complete.')
print(f'Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Remaining missing values: {df.isnull().sum().sum()}')
df.head(3)

## 4. Exploratory Data Analysis (EDA)

### 4.1 Revenue Trend Over Time

In [ ]:
monthly_rev = df.groupby('YearMonth')['Total Purchase Amount'].sum().reset_index()
monthly_rev['YearMonth_str'] = monthly_rev['YearMonth'].astype(str)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(monthly_rev['YearMonth_str'], monthly_rev['Total Purchase Amount'] / 1e6,
        marker='o', markersize=3, linewidth=1.8, color='steelblue')
ax.set_title('Monthly Revenue Trend (Jan 2020 – Sep 2023)', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue ($ Millions)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.1f}M'))
tick_step = max(1, len(monthly_rev) // 12)
ax.set_xticks(range(0, len(monthly_rev), tick_step))
ax.set_xticklabels(monthly_rev['YearMonth_str'][::tick_step], rotation=45, ha='right')
plt.tight_layout()
plt.savefig('fig_monthly_revenue.png', bbox_inches='tight')
plt.show()

### 4.2 Revenue by Product Category

In [ ]:
cat_rev = df.groupby('Product Category')['Total Purchase Amount'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
bars = axes[0].bar(cat_rev.index, cat_rev.values / 1e6,
                   color=['#4C72B0','#DD8452','#55A868','#C44E52'])
axes[0].set_title('Total Revenue by Product Category', fontweight='bold')
axes[0].set_ylabel('Revenue ($ Millions)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.3,
                 f'${bar.get_height():.1f}M', ha='center', va='bottom', fontsize=10)

# Pie chart
axes[1].pie(cat_rev.values, labels=cat_rev.index, autopct='%1.1f%%',
            colors=['#4C72B0','#DD8452','#55A868','#C44E52'], startangle=90)
axes[1].set_title('Revenue Share by Category', fontweight='bold')

plt.tight_layout()
plt.savefig('fig_category_revenue.png', bbox_inches='tight')
plt.show()

### 4.3 Distribution of Total Purchase Amount

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['Total Purchase Amount'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Order Value', fontweight='bold')
axes[0].set_xlabel('Total Purchase Amount ($)')
axes[0].set_ylabel('Frequency')

sns.boxplot(data=df, x='Product Category', y='Total Purchase Amount',
            palette='muted', ax=axes[1])
axes[1].set_title('Order Value by Category', fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Total Purchase Amount ($)')

plt.tight_layout()
plt.savefig('fig_order_distribution.png', bbox_inches='tight')
plt.show()

### 4.4 Age Distribution & Gender Split

In [ ]:
df_cust = df.drop_duplicates('Customer ID')[['Customer ID','Age','Age Group','Gender','Churn']]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_cust['Age'], bins=20, color='#4C72B0', edgecolor='white')
axes[0].set_title('Customer Age Distribution', fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Number of Customers')

gender_counts = df_cust['Gender'].value_counts()
axes[1].pie(gender_counts.values, labels=gender_counts.index,
            autopct='%1.1f%%', colors=['#4C72B0','#DD8452'], startangle=90)
axes[1].set_title('Gender Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig('fig_demographics.png', bbox_inches='tight')
plt.show()

### 4.5 Payment Method Usage

In [ ]:
pm = df['Payment Method'].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(pm.index, pm.values, color=['#4C72B0','#DD8452','#55A868'])
ax.set_title('Transactions by Payment Method', fontweight='bold')
ax.set_ylabel('Number of Transactions')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 500,
            f'{bar.get_height():,}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('fig_payment_methods.png', bbox_inches='tight')
plt.show()

### 4.6 Returns Distribution

In [ ]:
returns_by_cat = df.groupby('Product Category')['Returns'].mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(returns_by_cat.index, returns_by_cat.values,
              color=['#C44E52','#4C72B0','#55A868','#DD8452'])
ax.set_title('Return Rate by Product Category (%)', fontweight='bold')
ax.set_ylabel('Return Rate (%)')
ax.set_ylim(0, 70)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('fig_returns_by_category.png', bbox_inches='tight')
plt.show()

## 5. Key Performance Indicators (KPIs)

In [ ]:
total_revenue       = df['Total Purchase Amount'].sum()
total_transactions  = len(df)
unique_customers    = df['Customer ID'].nunique()
avg_order_value     = df['Total Purchase Amount'].mean()
return_rate         = df['Returns'].mean() * 100

# Customer-level churn
churn_rate = df.drop_duplicates('Customer ID')['Churn'].mean() * 100

# Avg transactions per customer
avg_txn_per_cust = total_transactions / unique_customers

# Customer Lifetime Value (CLV) = avg order value × avg transactions per customer
clv = avg_order_value * avg_txn_per_cust

print('=' * 50)
print('       BUSINESS KPI SUMMARY')
print('=' * 50)
print(f'  Total Revenue               : ${total_revenue:>15,.0f}')
print(f'  Total Transactions          : {total_transactions:>15,}')
print(f'  Unique Customers            : {unique_customers:>15,}')
print(f'  Avg Order Value (AOV)       : ${avg_order_value:>15,.2f}')
print(f'  Avg Transactions/Customer   : {avg_txn_per_cust:>15.2f}')
print(f'  Estimated CLV               : ${clv:>15,.2f}')
print(f'  Return Rate                 : {return_rate:>14.2f}%')
print(f'  Customer Churn Rate         : {churn_rate:>14.2f}%')
print('=' * 50)

## 6. Customer Segmentation & Behaviour Analysis

### 6.1 RFM — Customer-Level Feature Engineering

In [ ]:
snapshot_date = df['Purchase Date'].max() + pd.Timedelta(days=1)

rfm = df.groupby('Customer ID').agg(
    Recency  =('Purchase Date', lambda x: (snapshot_date - x.max()).days),
    Frequency=('Customer ID', 'count'),
    Monetary =('Total Purchase Amount', 'sum'),
    ReturnRate=('Returns', 'mean'),
    Age      =('Age', 'first'),
    Gender   =('Gender', 'first'),
    AgeGroup =('Age Group', 'first'),
    Churn    =('Churn', 'first')
).reset_index()

# RFM quintile scoring (5 = best)
rfm['R_Score'] = pd.qcut(rfm['Recency'],   5, labels=[5, 4, 3, 2, 1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
rfm['M_Score'] = pd.qcut(rfm['Monetary'],  5, labels=[1, 2, 3, 4, 5])
rfm['RFM_Score'] = rfm['R_Score'].astype(int) + rfm['F_Score'].astype(int) + rfm['M_Score'].astype(int)

# Segment based on RFM score
def rfm_segment(score):
    if score >= 13:
        return 'Champions'
    elif score >= 10:
        return 'Loyal'
    elif score >= 7:
        return 'At Risk'
    else:
        return 'Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(rfm_segment)
print('RFM Segmentation complete.')
print(rfm['Segment'].value_counts())
rfm[['Customer ID','Recency','Frequency','Monetary','RFM_Score','Segment','Churn']].head()

### 6.2 RFM Segment Distribution

In [ ]:
seg_counts = rfm['Segment'].value_counts()
seg_order  = ['Champions','Loyal','At Risk','Lost']
seg_counts = seg_counts.reindex(seg_order)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['#55A868','#4C72B0','#DD8452','#C44E52']
bars = axes[0].bar(seg_counts.index, seg_counts.values, color=colors)
axes[0].set_title('Customer Segments (RFM)', fontweight='bold')
axes[0].set_ylabel('Number of Customers')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 100,
                 f'{bar.get_height():,}', ha='center', va='bottom', fontsize=9)

seg_churn = rfm.groupby('Segment')['Churn'].mean().reindex(seg_order) * 100
bars2 = axes[1].bar(seg_churn.index, seg_churn.values, color=colors)
axes[1].set_title('Churn Rate by RFM Segment (%)', fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_ylim(0, 40)
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.3,
                 f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('fig_rfm_segments.png', bbox_inches='tight')
plt.show()

### 6.3 Revenue & Spending by Age Group and Gender

In [ ]:
age_rev = df.groupby('Age Group', observed=True)['Total Purchase Amount'].mean()
gender_rev = df.groupby('Gender')['Total Purchase Amount'].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bars = axes[0].bar(age_rev.index, age_rev.values, color=sns.color_palette('muted', len(age_rev)))
axes[0].set_title('Avg Order Value by Age Group', fontweight='bold')
axes[0].set_ylabel('Avg Order Value ($)')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 5,
                 f'${bar.get_height():.0f}', ha='center', va='bottom', fontsize=9)

bars2 = axes[1].bar(gender_rev.index, gender_rev.values, color=['#4C72B0','#DD8452'])
axes[1].set_title('Avg Order Value by Gender', fontweight='bold')
axes[1].set_ylabel('Avg Order Value ($)')
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 5,
                 f'${bar.get_height():.0f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('fig_age_gender_spending.png', bbox_inches='tight')
plt.show()

### 6.4 Preferred Payment Method by Age Group

In [ ]:
pm_age = df.groupby(['Age Group','Payment Method'], observed=True).size().unstack(fill_value=0)
pm_age_pct = pm_age.div(pm_age.sum(axis=1), axis=0) * 100

ax = pm_age_pct.plot(kind='bar', stacked=True, figsize=(10, 5),
                     color=['#4C72B0','#DD8452','#55A868'], edgecolor='white')
ax.set_title('Payment Method Preference by Age Group (%)', fontweight='bold')
ax.set_xlabel('Age Group')
ax.set_ylabel('Share of Transactions (%)')
ax.set_xticklabels(pm_age_pct.index, rotation=0)
ax.legend(title='Payment Method', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig('fig_payment_by_age.png', bbox_inches='tight')
plt.show()

## 7. Churn Analysis

### 7.1 Overall Churn Rate

In [ ]:
churn_counts = rfm['Churn'].value_counts().rename({0:'Retained', 1:'Churned'})
churn_rate_val = rfm['Churn'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].pie(churn_counts.values, labels=churn_counts.index,
            autopct='%1.1f%%', colors=['#55A868','#C44E52'], startangle=90,
            explode=[0, 0.05])
axes[0].set_title(f'Customer Churn Rate = {churn_rate_val:.1f}%', fontweight='bold')

churn_by_age = rfm.groupby('AgeGroup', observed=True)['Churn'].mean() * 100
axes[1].bar(churn_by_age.index, churn_by_age.values,
            color=sns.color_palette('muted', len(churn_by_age)))
axes[1].set_title('Churn Rate by Age Group (%)', fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_ylim(0, 30)
for i, v in enumerate(churn_by_age.values):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('fig_churn_overview.png', bbox_inches='tight')
plt.show()

### 7.2 Churn by Payment Method, Gender, and Category

In [ ]:
churn_pm     = df.groupby('Payment Method')['Churn'].mean().sort_values(ascending=False) * 100
churn_gender = rfm.groupby('Gender')['Churn'].mean().sort_values(ascending=False) * 100
churn_cat    = df.groupby('Product Category')['Churn'].mean().sort_values(ascending=False) * 100

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, data, title in zip(
    axes,
    [churn_pm, churn_gender, churn_cat],
    ['Churn Rate by Payment Method', 'Churn Rate by Gender', 'Churn Rate by Product Category']
):
    bars = ax.bar(data.index, data.values, color=sns.color_palette('muted', len(data)))
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Churn Rate (%)')
    ax.set_ylim(0, 30)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.2,
                f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('fig_churn_breakdown.png', bbox_inches='tight')
plt.show()

### 7.3 Churn vs Frequency & Monetary (RFM)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

churned   = rfm[rfm['Churn'] == 1]
retained  = rfm[rfm['Churn'] == 0]

axes[0].hist(retained['Frequency'], bins=15, alpha=0.6, color='#55A868', label='Retained', density=True)
axes[0].hist(churned['Frequency'],  bins=15, alpha=0.6, color='#C44E52', label='Churned',  density=True)
axes[0].set_title('Purchase Frequency: Churned vs Retained', fontweight='bold')
axes[0].set_xlabel('Number of Transactions')
axes[0].set_ylabel('Density')
axes[0].legend()

axes[1].hist(retained['Monetary'] / 1000, bins=20, alpha=0.6, color='#55A868', label='Retained', density=True)
axes[1].hist(churned['Monetary']  / 1000, bins=20, alpha=0.6, color='#C44E52', label='Churned',  density=True)
axes[1].set_title('Total Spend: Churned vs Retained', fontweight='bold')
axes[1].set_xlabel('Total Spend ($000s)')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.savefig('fig_churn_rfm.png', bbox_inches='tight')
plt.show()

print('Mean Frequency — Retained:', round(retained['Frequency'].mean(), 2),
      '  Churned:', round(churned['Frequency'].mean(), 2))
print('Mean Monetary  — Retained: $', round(retained['Monetary'].mean(), 0),
      '  Churned: $', round(churned['Monetary'].mean(), 0))

## 8. Machine Learning — Churn Prediction

### 8.1 Feature Engineering & Preprocessing

In [ ]:
# Build ML feature set from customer-level RFM data
ml_df = rfm[['Recency','Frequency','Monetary','ReturnRate','Age','Gender','Churn']].copy()

# Encode Gender
le = LabelEncoder()
ml_df['Gender_enc'] = le.fit_transform(ml_df['Gender'])
ml_df.drop(columns=['Gender'], inplace=True)

X = ml_df.drop(columns=['Churn'])
y = ml_df['Churn']

print('Feature matrix shape:', X.shape)
print('Class balance:')
print(y.value_counts(normalize=True).round(3))
X.head()

### 8.2 Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training set : {X_train.shape[0]:,} customers')
print(f'Test set     : {X_test.shape[0]:,} customers')

### 8.3 Logistic Regression Baseline

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_sc, y_train)

y_pred_lr = lr.predict(X_test_sc)
y_prob_lr = lr.predict_proba(X_test_sc)[:, 1]

print('=== Logistic Regression ===')
print(classification_report(y_test, y_pred_lr, target_names=['Retained','Churned']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob_lr):.4f}')

### 8.4 Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=20,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print('=== Random Forest ===')
print(classification_report(y_test, y_pred_rf, target_names=['Retained','Churned']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob_rf):.4f}')

### 8.5 Model Evaluation — ROC Curve & Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ROC Curves
for name, prob in [('Logistic Regression', y_prob_lr), ('Random Forest', y_prob_rf)]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
axes[0].plot([0,1],[0,1],'k--', label='Random')
axes[0].set_title('ROC Curve Comparison', fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(fontsize=9)

# Confusion Matrix — Logistic Regression
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_lr),
    display_labels=['Retained','Churned']
).plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Logistic Regression\nConfusion Matrix', fontweight='bold')

# Confusion Matrix — Random Forest
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_rf),
    display_labels=['Retained','Churned']
).plot(ax=axes[2], colorbar=False, cmap='Oranges')
axes[2].set_title('Random Forest\nConfusion Matrix', fontweight='bold')

plt.tight_layout()
plt.savefig('fig_model_evaluation.png', bbox_inches='tight')
plt.show()

### 8.6 Feature Importance (Random Forest)

In [ ]:
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(feat_imp.index, feat_imp.values * 100,
              color=sns.color_palette('muted', len(feat_imp)))
ax.set_title('Feature Importance — Random Forest (%)', fontweight='bold')
ax.set_ylabel('Importance (%)')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.2,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('fig_feature_importance.png', bbox_inches='tight')
plt.show()

print(feat_imp.round(4))

### 8.7 Cross-Validation — Robustness Check

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores_lr = cross_val_score(lr, X_train_sc, y_train, cv=cv, scoring='roc_auc')
cv_scores_rf = cross_val_score(rf, X_train, y_train, cv=cv, scoring='roc_auc')

print(f'Logistic Regression CV AUC: {cv_scores_lr.mean():.4f} ± {cv_scores_lr.std():.4f}')
print(f'Random Forest     CV AUC: {cv_scores_rf.mean():.4f} ± {cv_scores_rf.std():.4f}')

### 8.8 Discussion — Why Model Performance is Low

> **Note on ML Performance:**  
> Both models achieve ROC-AUC close to 0.50 (random baseline). This is **not a modelling error** — it reflects the structure of the dataset itself. The correlation analysis shows that **none of the available features (Recency, Frequency, Monetary, Returns, Age, Gender) correlate meaningfully with Churn** (all |r| < 0.01). Churn in this dataset appears to be **assigned independently of behavioural patterns**, suggesting the label may have been synthetically generated. Despite this, the models are correctly built and evaluated. The analytics sections (Sections 4–7) contain the genuine business value of this project.

## 9. Business Insights & Recommendations

In [ ]:
print('=== KEY BUSINESS INSIGHTS ===')
print()
print('1. CHURN RATE IS ~20%')
print(f'   - {rfm["Churn"].sum():,} out of {len(rfm):,} customers have churned.')
print(f'   - At avg CLV of ${clv:,.0f}, this represents ~${rfm["Churn"].sum() * clv:,.0f} in lost revenue potential.')
print()
print('2. REVENUE IS EVENLY DISTRIBUTED')
print('   - All 4 product categories contribute ~25% of revenue each.')
print('   - No single category dominates; diversification is a strength.')
print()
print('3. HIGH RETURN RATE OF ~50%')
print('   - Half of all transactions involve a return.')
print('   - This is a significant cost driver and satisfaction risk.')
print()
print('4. PAYMENT METHODS ARE WELL-BALANCED')
print('   - Credit Card, PayPal, and Cash share transactions almost equally.')
print()
print('5. YOUNG AND OLDER CUSTOMERS CHURN SLIGHTLY MORE')
print('   - Age groups 18-25 and 56-70 show marginally higher churn rates.')
print()
print('6. RFM SEGMENTATION REVEALS ACTIONABLE GROUPS')
print('   - Champions and Loyal customers should be rewarded.')
print('   - At Risk and Lost customers need re-engagement campaigns.')
print()
print('=== BUSINESS RECOMMENDATIONS ===')
print()
print('R1: Launch a Loyalty Programme targeting Champions and Loyal segments.')
print('R2: Run win-back email/SMS campaigns for At Risk and Lost customers.')
print('R3: Investigate and reduce the ~50% return rate through better product')
print('    descriptions, size guides, and quality checks.')
print('R4: Design age-specific retention offers for 18-25 and 56-70 groups.')
print('R5: Monitor monthly revenue for downward trends as an early churn signal.')
print('R6: Collect richer behavioural data (session time, support tickets,')
print('    discount usage) to build a more predictive churn model in future.')

---
## Summary

| KPI | Value |
|-----|-------|
| Total Revenue | ~$681M |
| Total Transactions | 250,000 |
| Unique Customers | 49,661 |
| Avg Order Value | ~$2,725 |
| Avg Transactions/Customer | ~5 |
| Return Rate | ~50% |
| Customer Churn Rate | ~20% |

**Key Takeaway:** The business is generating healthy, diversified revenue. The two critical risk areas are the **20% churn rate** and the **~50% return rate**, both of which require targeted operational and marketing responses.